### This notebook will help you to compare your data with m/e/me-type expressions generated by the model. 
- We will use the CCFv3 template to generate a whole brain slice from the brain region your samples are from.
- You can chose any cell type.

In [1]:
from pathlib import Path

current = Path().resolve()

while not (current / ".git").exists():
    current = current.parent

PROJECT_ROOT = current

In [2]:
PROJECT_ROOT

PosixPath('/home/cs/git/probabilistic_mapping_extention')

In [3]:
# %matplotlib widget

import os
import pandas as pd
import nrrd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from tqdm import tqdm

In [4]:
# Define the worker functions

def process_CCFv3a(CCFv3a, label_to_cluster, label_numbers):

    new_CCFv3a = np.full_like(CCFv3a, np.nan)

    for label in label_numbers:
        new_CCFv3a[CCFv3a == label] = label_to_cluster[label]
        
    # Retain existing np.nan values from the original CCFv3a
    nan_mask = np.isnan(CCFv3a)
    new_CCFv3a[nan_mask] = np.nan
    
    # Set 0 where values have not been populated
    new_CCFv3a[(~nan_mask) & (new_CCFv3a == np.nan)] = 0
    
    return new_CCFv3a

def create_segmentation_image(segmentation_array, colors_dict=None):

    # Allow float if it contains NaN
    if not (np.issubdtype(segmentation_array.dtype, int) or
            np.issubdtype(segmentation_array.dtype, float)):
        raise TypeError("Segmentation array must contain integers or NaN.")

    if colors_dict is None:
        colors_dict = {0: np.array([0, 0, 0])}

    segmentation_img = np.zeros((*segmentation_array.shape, 3), dtype=np.uint8)

    # Remove NaN for label discovery
    valid_mask = ~np.isnan(segmentation_array)
    valid_labels = np.unique(segmentation_array[valid_mask]).astype(int)

    for lb in valid_labels:

        if lb not in colors_dict:
            color = np.random.randint(255, size=3)
            colors_dict[lb] = color
        else:
            color = colors_dict[lb]

        segmentation_img[segmentation_array == lb] = color

    return segmentation_img, colors_dict

def data_visualization(data_arrays, is_categorical, titles=None):

    @widgets.interact(
        axis=widgets.Dropdown(
            options=[('Coronal', 0), ('Transversal', 1), ('Sagittal', 2)],
            value=1,
            description='Axis:',
        ),
    )
    def axis_view(axis):

        data_reoriented = [np.moveaxis(arr, axis, 0) for arr in data_arrays]

        @widgets.interact(
            slice_index=widgets.IntSlider(
                min=0,
                max=data_reoriented[0].shape[0] - 1,
                description='Section:',
            ),
        )
        def slice_view(slice_index):

            fig, ax = plt.subplots(1, len(data_reoriented), figsize=(25, 20))

            if len(data_reoriented) == 1:
                ax = [ax]  # make iterable

            displayed_images = []

            for i, (arr, is_cat) in enumerate(zip(data_reoriented, is_categorical)):

                ax[i].axes.get_xaxis().set_ticks([])
                ax[i].axes.get_yaxis().set_ticks([])

                if titles is not None:
                    ax[i].set_title(titles[i])

                slice_data = arr[slice_index]

                if is_cat:
                    img = create_segmentation_image(slice_data, color_map)[0]
                else:
                    img = slice_data

                im = ax[i].imshow(img)
                displayed_images.append((im, slice_data))

            # ---- HOVER FUNCTIONALITY ----
            text = fig.text(0.5, 0.01, "", ha="center")

            def on_move(event):
                for axis_obj, (im, original_slice) in zip(ax, displayed_images):
                    if event.inaxes == axis_obj:
                        x, y = int(event.xdata), int(event.ydata)

                        if (0 <= y < original_slice.shape[0] and
                            0 <= x < original_slice.shape[1]):

                            pixel_value = original_slice[y, x]
                            text.set_text(
                                f"x={x}, y={y} | value={pixel_value}"
                            )
                            fig.canvas.draw_idle()
                        return

            fig.canvas.mpl_connect("motion_notify_event", on_move)

            plt.show()



def process_multiple_celltypes(celltype_list, combined_df, CCFv3a):
    """
    Process multiple cell types using CCFv3a and label mapping.

    Parameters
    ----------
    celltype_list : list of str
        List of cell type column names in combined_df.
    combined_df : pd.DataFrame
        DataFrame containing 'label_numbers' and cell type columns.
    CCFv3a : np.array or appropriate object
        The original brain atlas array to process.

    Returns
    -------
    processed_dict : dict
        Dictionary mapping cell type string → processed array.
    """

    processed_dict = {}

    label_numbers = combined_df['label_numbers'].values

    for celltype in tqdm(celltype_list, desc="Processing cell types"):

        cluster_values = combined_df[celltype].values
        label_to_cluster = dict(zip(label_numbers, cluster_values))

        # Process using your existing function
        processed_array = process_CCFv3a(CCFv3a, label_to_cluster, label_numbers)

        processed_dict[celltype] = processed_array

    return processed_dict

#### Load data (we created this dataframe from m/e/me-type densities). 
- Values are in cell/mm3, except for the last colum (label_numbers) which stands for the lable id of the region in the index.
- We stored the way to reproduce this csv file here: csaba_create_all_celltype_df.ipynb

In [6]:
celltype_folder = PROJECT_ROOT / "extension/data/"
file_path = os.path.join(celltype_folder, "all_celltypes_composition_with_unassigned_regions.csv")

combined_df = pd.read_csv(file_path, index_col=0)
combined_df.shape

(719, 530)

In [7]:
combined_df.head()

,bAC,bIR,bNAC,bSTUT,cAC,cADpyr,cIR,cNAC,cSTUT,dNAC,...,IN_DEND_7_AX_0|dSTUT,IN_DEND_7_AX_2|dSTUT,IN_DEND_7_AX_3|dSTUT,IN_DEND_7_AX_8|dSTUT,IN_DEND_8_AX_8|dSTUT,IN_DEND_9_AX_1|dSTUT,IN_DEND_9_AX_2|dSTUT,IN_DEND_9_AX_5|dSTUT,IN_DEND_9_AX_8|dSTUT,label_numbers
AAA,13642.960359,7388.022628,12678.735695,2939.319168,21618.896401,11435.747609,2537.236585,6218.253955,3614.333926,1752.112101,...,109.751091,388.660591,60.923194,68.018737,11.714693,14.186047,2.240268,35.851274,37.195722,23
ACAd1,2799.640015,1121.977282,1715.824896,190.197327,2006.112983,7037.917440,146.280937,513.474338,224.182900,111.887536,...,0.510173,4.546404,11.648074,5.622710,20.888031,23.041590,1.049859,0.000000,5.498831,935
ACAd23,13098.155452,5396.225645,8453.156564,1068.118307,10729.650630,63898.354311,813.971276,2714.385602,1278.890566,675.691549,...,1.805363,37.214394,14.597905,51.716315,150.117788,39.591711,1.909610,0.000000,7.995749,211
ACAd5,6960.326343,3182.832919,5021.448421,866.551337,7442.602045,23646.689691,719.415675,2128.219118,1037.014350,488.061687,...,6.613080,27.502729,27.959385,41.723606,28.293837,31.184562,0.312694,0.000000,2.498040,1015
ACAd6a,11254.728651,4587.390288,6578.258284,734.551204,6913.793904,47203.835802,680.841571,2201.730703,824.338643,233.284512,...,1.736625,10.005915,12.827652,18.001872,99.345673,19.296173,0.972735,0.134379,21.993218,919


Next we load the suitable common coordinate framework version, e.g. the CCFv3a from the paper. This will create a 3D volume for visualisation for the cell types.

In [8]:
file_path = PROJECT_ROOT / "extension/annotation_25_2022_CCFv3a.nrrd"

CCFv3a, color_map = nrrd.read(file_path)

# Replace all zeros in CCFv3a with np.nan
CCFv3a = np.where(CCFv3a == 0, np.nan, CCFv3a)

Next, we can select any number of cell types to visualize in 3D. Here we select 1 M-, 1 E-, and 1 ME-type

In [14]:
celltypes = ['IN_DEND_0_AX_0', 'bAC', 'IN_DEND_7_AX_0|dSTUT']  # your cell type columns
processed_arrays = process_multiple_celltypes(celltypes, combined_df, CCFv3a)

Processing cell types: 100%|█████████████████████████████████████████████████████| 3/3 [07:31<00:00, 150.60s/it]


Optional: Next, we can select a name for them so we can distinguish between them in the 3D visualisation.

In [15]:
# Access processed array for a cell type:
brain_IN_DEND_0_AX_0 = processed_arrays['IN_DEND_0_AX_0']
brain_bAC = processed_arrays['bAC']
brain_IN_DEND_7_AX_0dSTUT = processed_arrays['IN_DEND_7_AX_0|dSTUT']

#### Visualization

Finally, we can call the data_visualization() function to visalize the brain volumes. 
- You can add your own to the list.
- If you want to add an annotation volume, set its value to TRUE so matplotlib can properly add random colurs to the image or select its own color_map above.
- You can select Axial, Transversal, and Sagittal views.
- It's possible to just visualize a 2D brain slice of yours next the 3D volumes.
- If you have %matplotlib widget you can use JavaScript to visalize the volumes and get pixel values and coordinates by hovering the mouse over the images. 

In [16]:
data_visualization([brain_IN_DEND_0_AX_0, brain_bAC, brain_IN_DEND_7_AX_0dSTUT, CCFv3a], [False, False, False, True])

interactive(children=(Dropdown(description='Axis:', index=1, options=(('Coronal', 0), ('Transversal', 1), ('Sa…